In [ ]:
# Suppress warnings from sentence-transformers and other libraries
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load pretrained sentence embedding model from HuggingFace
# all-MiniLM-L6-v2 converts text to 384-dimensional vectors
# Downloaded once and cached locally for subsequent runs
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(text: str):
    """
    Convert a text string into a 384-dimensional embedding vector.
    Similar texts will have vectors that are close together in space.
    Used to compare user preferences with movie descriptions.
    """
    return model.encode([text])[0]

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
# src.embeddings import not used - notebook-based project
# embed() is defined directly above

def preference_strength(genres, mood, favorite):
    """
    Calculate how strong a user's preferences are.
    More genres, a defined mood, and a favorite movie
    all increase the strength score.
    Used to compute relative weights in couple mode.
    """
    score = len(genres) * 0.3 # each genre adds 0.3
    score += 1.0 if mood else 0 # mood adds 1.0
    score += 1.5 if favorite else 0 # favorite movie adds 1.5
    return max(score, 0.1) # minimum 0.1 to avoid division by zero

def compute_weights(user_a, user_b):
    """
    Compute alpha and beta weights for couple fusion.
    The user with stronger preferences gets a higher weight.
    Returns two floats that sum to 1.0 (e.g. 0.6 and 0.4).
    """
    sa = preference_strength(user_a["genres"], user_a["mood"], user_a["favorite"])
    sb = preference_strength(user_b["genres"], user_b["mood"], user_b["favorite"])
    total = sa + sb
    return round(sa / total, 2), round(sb / total, 2)

def fuse_vectors(vec_a, vec_b, alpha, beta):
    """
    Combine two user preference vectors into one fused vector.
    alpha and beta are weights that must sum to 1.0.
    Result is normalized to unit length for cosine similarity.
    Example: alpha=0.6, beta=0.4 means Person A has more influence.
    """
    fused = alpha * np.array(vec_a) + beta * np.array(vec_b)
    return fused / np.linalg.norm(fused)

def compatibility_score(vec_a, vec_b):
    """
    Measure how similar two users' preferences are.
    Uses cosine similarity between their preference vectors.
    Returns a percentage score and a human-readable label.

    Score ranges:
    75%+    Movie soulmates
    60-74%  Great match
    40-59%  Some overlap
    below 40%  Opposites - challenge mode
    """
    score = cosine_similarity([vec_a], [vec_b])[0][0]
    pct = round(float(score) * 100)
    if pct >= 75: label = "Movie soulmates"
    elif pct >= 60: label = "Great match"
    elif pct >= 40: label = "Some overlap"
    else: label = "Opposites - challenge mode"
    return pct, label

In [ ]:
# Test 1 — compute_weights: stronger preferences get higher weight
user_a = {"genres": ["Action", "Sci-Fi"], "mood": "Adventurous", "favorite": "Interstellar"}
user_b = {"genres": ["Romance"], "mood": "Romantic", "favorite": None}

alpha, beta = compute_weights(user_a, user_b)
print("Test 1 — Compute weights:")
print(f"Person A weight: {alpha}")
print(f"Person B weight: {beta}")
print(f"Weights sum to 1: {round(alpha + beta, 2) == 1.0}")
print(f"Person A stronger (more genres + favorite): {alpha > beta}")
print()

In [ ]:
# Test 2 — compatibility_score: similar users score higher than opposites
#%run 05_embeddings.ipynb

vec_a = embed("I love action sci-fi adventure movies")
vec_b_similar = embed("I enjoy action films and science fiction")
vec_b_opposite = embed("I only watch romantic comedies and love stories")

score_similar, label_similar = compatibility_score(vec_a, vec_b_similar)
score_opposite, label_opposite = compatibility_score(vec_a, vec_b_opposite)

print("Test 2 — Compatibility score:")
print(f"Similar users: {score_similar}% - {label_similar}")
print(f"Opposite users: {score_opposite}% - {label_opposite}")
print(f"Similar scores higher than opposite: {score_similar > score_opposite}")